# 04 — Ensemble learning and frozen-protocol evaluation

This notebook orchestrates reusable Phase 4 modules; **all execution flags default to False**. It does not train models, construct ImageNet backbones, download images or regenerate splits. Real `.keras` checkpoints and original frozen training metadata must be provided privately; the committed `docs/experiments/` summaries contain no weights. No experimental Phase 4 results are bundled.

## 1. Independent Kaggle / Colab / local setup
Configure paths below. Kaggle is checked before Colab; Python 3.10–3.13 is supported. Use the existing training environment (TensorFlow 2.20.x / 2.21.x); installation is optional and requires a kernel restart. On Colab, mount Drive before accessing persistent session files. On Kaggle, attach the full private training run with checkpoints, contracts/states and manifest. Save/export evaluation outputs before leaving the runtime.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import copy
import json

RUN_PLATFORM_SETUP = False
CLONE_REPOSITORY = False
INSTALL_DEPENDENCIES = False
VERIFY_CHECKPOINTS = False
RUN_VALIDATION = False
RUN_FINAL_TEST = False
SHOW_SAVED_TABLES = False

PLATFORM = "kaggle" if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").is_dir()) else "colab" if Path("/content").is_dir() else "local"
default_repo = "/kaggle/working/glaucoma_detection" if PLATFORM == "kaggle" else "/content/glaucoma_detection" if PLATFORM == "colab" else str(Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent)
REPO_ROOT = Path(os.environ.get("PROJECT_REPO", default_repo))
REPO_URL = "https://github.com/VDJ2824/glaucoma_detection.git"
REPO_REF = "main"  # prefer a reviewed commit for reproducibility
BUNDLE_ROOT = Path("/kaggle/input/your-private-training-run")  # EDIT: run containing checkpoints/manifests/environment
if PLATFORM == "kaggle":
    default_session = BUNDLE_ROOT / "environment/session.json"
    default_output = Path("/kaggle/working/artifacts")
elif PLATFORM == "colab":
    default_output = Path("/content/drive/MyDrive/glaucoma_detection/artifacts")
    default_session = default_output / "runs/development/environment/session.json"
else:
    default_output = REPO_ROOT / "artifacts"
    default_session = default_output / "runs/development/environment/session.json"
SESSION_FILE = os.environ.get("GLAUCOMA_SESSION", str(default_session))
MANIFEST_CSV = BUNDLE_ROOT / "manifests/acrima_metadata.csv" if PLATFORM == "kaggle" else None
CHECKPOINT_ROOT = BUNDLE_ROOT / "checkpoints/core" if PLATFORM == "kaggle" else None
# Optional per-model bases containing <model>/<stage>/; leave None for a common base/session setting.
CHECKPOINT_ROOTS = None
# To decode original Kaggle manifest paths on another machine, set the CURRENT flat image directory.
# Keep the original CSV unchanged; this affects only in-memory physical path lookup.
FLAT_IMAGE_ROOT = None
OUTPUT_ROOT = default_output
EVALUATION_NAME = "acrima_phase4_v1"
if RUN_VALIDATION and RUN_FINAL_TEST:
    raise ValueError("Run validation alone, review the frozen configuration, then disable it before final test opt-in.")


In [ ]:
if RUN_PLATFORM_SETUP:
    if not (3, 10) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError("Use Python 3.10–3.13.")
    if PLATFORM == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
    if CLONE_REPOSITORY:
        if REPO_ROOT.exists():
            raise FileExistsError("Existing clone preserved; inspect it rather than resetting it.")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT/"requirements-training.txt")], cwd=REPO_ROOT, check=True)
        raise RuntimeError("Restart the kernel, disable installation, then rerun.")
if not (REPO_ROOT/"src/glaucoma").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the existing project clone.")
sys.path.insert(0, str(REPO_ROOT/"src"))


## 2. Load saved session and predefine the selection policy

Do not change trained model architectures, preprocessing, seed, labels or split assignments. The manifest must be the original canonical CSV used by training, including its original stored paths: those rows are bound to each training contract. `FLAT_IMAGE_ROOT` permits decoding from a relocated flat directory **without editing that CSV**. An already rewritten path manifest whose training hash no longer matches is rejected; no contract hash is guessed or bypassed.

Policy: finite 0.05-step simplex grid plus exact equal weights (232 candidates), minimum validation BCE with clipping epsilon 1e-7. Loss ties within 1e-12 prefer closest-to-equal weights, then lexicographic weights in the explicit core-model order. The classification threshold is **predefined at 0.5**, not optimized. Majority voting uses the same threshold per model; a three-model hard vote has no tie. No test result may influence these choices.

In [ ]:
from glaucoma.utils.notebook_runtime import load_session
from glaucoma.ensemble.predictions import MEMBERS
from glaucoma.ensemble.weights import search_protocol
from glaucoma.evaluation.checkpoints import verify_selected_checkpoints
from glaucoma.evaluation.workflow import run_directory, run_validation, run_final_test, load_frozen

cfg = copy.deepcopy(load_session(SESSION_FILE))
manifest_path = Path(MANIFEST_CSV or cfg["metadata"]["output"])
checkpoint_roots = CHECKPOINT_ROOTS or ({name: str(CHECKPOINT_ROOT) for name in MEMBERS} if CHECKPOINT_ROOT is not None else None)
PROTOCOL = search_protocol(grid_denominator=20, threshold=0.5)
selection_records = {
    "mobilenet_v2": REPO_ROOT/"docs/experiments/mobilenet_v2_kaggle_2026-11-26/metrics/selection.json",
    "efficientnet_b0": REPO_ROOT/"docs/experiments/efficientnet_b0_kaggle_2026-11-27/metrics/selection.json",
    "resnet50": REPO_ROOT/"docs/experiments/resnet50_kaggle_2026-11-27/metrics/selection.json",
}
directory = run_directory(cfg, EVALUATION_NAME, OUTPUT_ROOT)
print("Manifest:", manifest_path)
print("Evaluation artifacts:", directory)
print("Member order:", MEMBERS)
print(json.dumps(PROTOCOL, indent=2))


## 3. Verify selected private checkpoints

Optional `VERIFY_CHECKPOINTS=True` checks both completed stages, actual `best.keras` files, state/contract digests, original training/validation manifest hash, input/head/preprocessing compatibility and the committed experiment's selection records. The Phase 3 selection/recovery utilities remain the authority. Model names, feature shapes, GAP/dropout/sigmoid head and inference graph are checked when loaded during prediction. Never substitute summary metrics for missing checkpoints. No weights are downloaded or rebuilt here.

In [ ]:
if VERIFY_CHECKPOINTS:
    verified = verify_selected_checkpoints(cfg, manifest_path, selection_records, checkpoint_roots)
    print(json.dumps(verified, indent=2))
else:
    print("Checkpoint verification disabled; no checkpoint files read.")


## 4. Validation inference, voting, predefined weight search and freeze

Enable only `RUN_VALIDATION`. Models load sequentially to limit memory and are called with `training=False`; the unchanged model-specific preprocessing runs on validation images without augmentation. Individual probability CSVs and aligned ground truth use dataset/image IDs, with duplicate/missing/mislabeled rows rejected. The last partial batch is supported.

The shared workflow saves candidate weights/losses, validation metrics/disagreement/plots and **frozen_ensemble.json**. It binds weights, threshold, policy, manifest hash, preprocessing/physical-root configuration, checkpoint byte hashes/contracts and model graphs. Existing runs are not overwritten. No test images or test labels are used for the weight search. The CSV must be read to locate validation rows and hash the frozen file; test rows are not inspected for labels, scored or used in the objective.

In [ ]:
validation_result = None
if RUN_VALIDATION:
    validation_result = run_validation(cfg, manifest_path, enabled=True,
        name=EVALUATION_NAME, output_root=OUTPUT_ROOT, expected_records=selection_records,
        checkpoint_roots=checkpoint_roots, flat_image_root=FLAT_IMAGE_ROOT, protocol=PROTOCOL)
    print(json.dumps(validation_result, indent=2))
else:
    print("Validation inference and ensemble optimization disabled.")


## 5. Review the frozen record before any test access

Optionally enable `SHOW_SAVED_TABLES` to inspect actual saved validation tables and the frozen record. In a separate deliberate step, disable validation and opt into final test evaluation. A freeze cannot be overwritten or retuned in this run, and the final workflow checks it before reading the test partition. Changing manifest contents, physical roots, checkpoints, preprocessing or threshold invalidates the run.

Boundary weights may select one model entirely. Validation selection is not independent evidence that weighted voting improves generalization. Do not try different grids/thresholds after examining test metrics.

In [ ]:
if SHOW_SAVED_TABLES:
    import pandas as pd
    print(json.dumps(load_frozen(directory), indent=2))
    print(pd.read_csv(directory/"validation/metrics_comparison.csv").to_string(index=False))
else:
    print("Saved validation inspection disabled.")


## 6. Final untouched-test evaluation — separate opt-in

Enable `RUN_FINAL_TEST=True` with `RUN_VALIDATION=False` only after freezing the protocol. The workflow rechecks manifest/checkpoint/contract fingerprints and the inference configuration, records `test_started.json`, then loads the test partition. All three individuals, majority voting, equal soft voting and weighted soft voting use the same aligned test images and frozen threshold.

Metrics: accuracy, precision, recall, specificity, F1, exact score-based ROC-AUC and clipped probability BCE. Hard majority has no probability score; its ROC-AUC/BCE are explicitly not applicable. Undefined denominators/single-class AUC are represented by null with reasons. Test execution is single-attempt per run; a failure after the marker is written must be investigated without changing the frozen protocol or using partial results to retune.

In [ ]:
test_result = None
if RUN_FINAL_TEST:
    test_result = run_final_test(cfg, manifest_path, enabled=True,
        name=EVALUATION_NAME, output_root=OUTPUT_ROOT, expected_records=selection_records,
        checkpoint_roots=checkpoint_roots, flat_image_root=FLAT_IMAGE_ROOT)
    print(json.dumps(test_result, indent=2))
else:
    print("Final test evaluation disabled. No test images accessed.")


## 7. Artifacts, plots and limitations

When explicitly run, the session's persistent artifact root receives `phase4/<EVALUATION_NAME>/`: predefined/frozen protocol records; validation/test individual and aligned scores; all strategy labels/scores; metrics JSON/CSV; confusion matrices; ROC curves/points; per-image and pairwise disagreement summaries. View the saved PNGs and tables directly. Colab outputs must be on mounted Drive; Kaggle working outputs must be saved/exported privately. Retinal images/weights/prediction tables are not committed automatically.

Image-level splitting does not establish patient independence, and pixel/path checks do not establish absence of near duplicates. This single ACRIMA evaluation cannot establish external generalization, clinical validity or superiority over an individual model. The same validation partition already supported checkpoint selection, so optimized ensemble validation metrics are selection-biased. Actual held-out test inference has **not** been run by implementation tests; all automated tests use synthetic data. See [Phase 4 contracts and execution guide](../docs/phase4-ensemble-evaluation.md).